<a href="https://colab.research.google.com/github/LemdjoM/Arduino/blob/master/Assignements/Part%204/Assignment_part_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DSML investigation

You are part of the Suisse Impossible Mission Force, or SIMF for short. You need to uncover a rogue agent that is trying to steal sensitive information.

Your mission, should you choose to accept it, is to find that agent before stealing any classified information. Good luck!

# Assignement part four

#### Identifying the suspects' credit score
We received informations that the rogue agent has a *good* credit score.

Our spies at SIMF have managed to collect financial information relating to our suspects as well as a training dataset.

Create a Neural Network over the training dataset `df` to identify which of the suspects have a *standard* `Credit_Mix`.


## Getting to know our data

* `Age`: a user's age

* `Occupation`: a user's employment field

* `Annual_Income`: a user's annual income

* `Monthly_Inh_Salary`: the calculated salary received by a given user on a monthly basis

* `Num_Bank_Accounts`: the number of bank accounts possessed by a given user

* `Num_Credit_Cards`: the number of credit cards a given user possesses

* `Interest_Rate`: The interest rate on those cards (if multiple then it's the average)

* `Num_of_Loans`: The number of loans of each user

* `Delay_from_due_date`: payment tardiness of user

* `Num_of_Delayed_Payment`: the count of delayed payments

* `Changed_Credit_Limit`: NaN

* `Num_Credit_Inquiries`: NaN

* `Credit_Mix`: The user's credit score

* `Outsting_Debt`: Outstanding debt

* `Credit_Utilization_Ratio`: the percentage of borrowed money over borrowing allowance

* `Payment_of_Min_Amount`: does the user usually pay the minimal amount (categorical)

* `Total_EMI_per_month`: Monthly repayments to be made

* `Amount_invested_monthly`: The amount put in an investment fund by the user on a monthly basis

* `Payment_Behaviour`: the user's payment behavior (categorical)

* `Monthly_Balance`: The user's end of the month balance

* `AutoLoan`: If the user has an active loan for their vehicle

* `Credit-BuilderLoan`: If the user has a loan to increase their credit score

* `DebtConsolidationLoan`, `HomeEquityLoan`, `MortgageLoan`, `NotSpecified`, `PaydayLoan`, `PersonalLoan`, `StudentLoan`: different types of loans (categorical features)


In [ ]:
# Import required packages

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler

%matplotlib inline

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/michalis0/DataScience_and_MachineLearning/master/Assignements/Part%204/data/train_classification.csv", index_col='Unnamed: 0').dropna()
suspects = pd.read_csv("https://raw.githubusercontent.com/michalis0/DataScience_and_MachineLearning/master/Assignements/Part%204/data/suspects.csv", index_col='Unnamed: 0').dropna()
suspects.rename(columns={"Payment_Behaviour": "le_Payment_Behaviour", "Payment_of_Min_Amount": "le_Payment_of_Min_Amount"}, inplace=True)

In [ ]:
display(df.head())
print(df.shape)
display(suspects.head())
print(suspects.shape)

In [ ]:
df["Credit_Mix"].unique()

# 1. Preparing the data
## 1.1 Data cleaning
Consider the dataset loaded into the DataFrame `df` which is *train_classification.csv*. We aim to preprocess this data for model training.

Begin by encoding the categorical variables:
- Apply One-Hot Encoding to `Occupation`
- Apply Label Encoding to `Payment_of_Min_Amount` and `Payment_Behaviour`

*Note: To clearly distinguish your newly encoded columns, especially for label encoding, consider renaming them with a prefix. Please, use `le_Payment_of_Min_Amount` and `le_Payment_Behaviour` for the label-encoded new columns.*

In [ ]:
# Your code here
#df_encoded = pd.get_dummies(df, columns=['Occupation'], prefix='Occupation')
#label_encoder = LabelEncoder()
#df_encoded['le_Payment_of_Min_Amount'] = label_encoder.fit_transform(df_encoded['Payment_of_Min_Amount'])
#df_encoded['le_Payment_Behaviour'] = label_encoder.fit_transform(df_encoded['Payment_Behaviour'])

# Creating df_label
#display(df_encoded.head())

# Label encoding
df_encoded = df.copy()
le_payment_min = LabelEncoder()
le_payment_behaviour = LabelEncoder()
df_encoded['le_Payment_of_Min_Amount'] = le_payment_min.fit_transform(df_encoded['Payment_of_Min_Amount'])
df_encoded['le_Payment_Behaviour'] = le_payment_behaviour.fit_transform(df_encoded['Payment_Behaviour'])
#df = df.drop(['Payment_of_Min_Amount', 'Payment_Behaviour'], axis=1)

# One-hot encoding for Occupation
ohe = OneHotEncoder(sparse_output=False, drop='first')
ohe_encoded = ohe.fit_transform(df_encoded[['Occupation']])
ohe_columns = ohe.get_feature_names_out(['Occupation'])
df_ohe = pd.DataFrame(ohe_encoded, columns=ohe_columns, index=df_encoded.index)
df_encoded = pd.concat([df_encoded.drop(['Occupation'], axis=1), df_ohe], axis=1)

df_encoded.shape



 After encoding, integrate the newly encoded columns back into a new DataFrame named `df_encoded`, and remove the original columns `Occupation`, `Payment_of_Min_Amount`, and `Payment_Behaviour` to avoid redundancy.

In [ ]:
# Your code here
# Remove the original categorical columns
#df_encoded = df_encoded.drop(['Occupation', 'Payment_of_Min_Amount', 'Payment_Behaviour'], axis=1)
df_encoded = df_encoded.drop(['Payment_of_Min_Amount', 'Payment_Behaviour'], axis=1)
df_encoded.shape

Finally, display the first few rows of `df_encoded` to verify that the encodings are correctly implemented. This prepared DataFrame will be used for subsequent model training.

In [ ]:
# Your code here
df_encoded.head()

## 1.2 Dataset splitting and rescaling

To effectively train and validate our model, it is crucial to properly prepare and partition the data. Follow these steps to preprocess and split the dataframe df_encoded into training and test subsets, ensuring that our model can generalize well to new data:

- Set `X` as all columns except `Credit_Mix` and `y` as the dependent feature `Credit_Mix`.
- Apply manually Label Encoding to `y` using the function `.map`and encoding with the following : Good=2, Standard=1, Bad=0.
- Use a `random_state` of 42 to split `X` and the encoded `y` into training (80%) and test sets (20%).
- Normalize `X` using `MinMaxScaler()`.

In [ ]:
# Your code here
y = df_encoded['Credit_Mix']  # Single brackets to get a Series
X = df_encoded.drop('Credit_Mix', axis = 1)

# Apply manual Label Encoding to y using map()
credit_mix_mapping = {'Bad': 0, 'Standard': 1, 'Good': 2}
y_encoded = y.map(lambda x: credit_mix_mapping.get(x, x))
#Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

#Normalize X using MinMaxScaler()
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


### 1.2.2 Final touches
Convert the features to torch tensors of type `torch.float` and the labels (dependent variables) to torch tensors of type `torch.long`.

In [ ]:
#Your code here
# Convert features to torch.float
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float)

# Convert labels to torch.long
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

print(X_train_tensor.size(), y_train_tensor.size())
print(X_test_tensor.size(), y_test_tensor.size())


# 2 Model preparation:

## 2.1 Define a Neural network model and instantiate it.
Define your neural network model as a class in PyTorch, extending from nn.Module. In the `__init__` method, initialize a linear layer using `nn.Linear()` with specified input and output sizes, and set up `nn.ReLU()` for activation. Implement the forward method to describe how data passes through this layer during the network's forward computation.

Set the following parameters:
* `hidden layer` : 1
* `activation function` : ReLU

In [ ]:
# Your code here
#Define the Neural Network model
class CreditMixClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CreditMixClassifier, self).__init__()
        # Define the first linear layer (input to hidden)
        self.linear1 = nn.Linear(input_size, hidden_size)
        # Define ReLU activation
        self.relu = nn.ReLU()
        # Define the output linear layer (hidden to output)
        self.linear2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Pass input through the first linear layer
        x = self.linear1(x)
        # Apply ReLU activation
        x = self.relu(x)
        # Pass through the output linear layer
        x = self.linear2(x)
        return x

# Instantiate the model
input_size = X_train_tensor.shape[1]  # Number of features
hidden_size = 64  # Arbitrary choice, can be tuned
output_size = 3   # Number of classes (Bad: 0, Standard: 1, Good: 2)

model = CreditMixClassifier(input_size, hidden_size, output_size)

# Verify the model
print("Model architecture:\n", model)
print("\nInput size:", input_size)
print("Hidden size:", hidden_size)
print("Output size:", output_size)

Set `D_in` to the number of features in `X_train` and `D_out` to the number of target variables in `y_train`, then print these dimensions to verify their values.

In [ ]:
# Your code here
# Set D_in and D_out
D_in = X_train.shape[1]  # Number of features in X_train
D_out = len(y_train.unique())  # Number of unique classes in y_train
print(D_in, D_out)




Initialize the `Net` model with the specified input size `D_in`, 150 hidden units, and output size `D_out`.

In [ ]:
# Your code here
hidden_size = 150  # Specified number of hidden units
model = CreditMixClassifier(input_size=D_in, hidden_size=hidden_size, output_size=D_out)

# Verify the model
print("\nModel architecture:\n", model)
print("Input size (D_in):", D_in)
print("Hidden size:", hidden_size)
print("Output size (D_out):", D_out)

Let's calculate now how many parameters we have in the model.


In [ ]:
# Your code here
# Calculate the number of parameters
pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable parameters:", pytorch_total_params)

**Q1. How many parameters does your model have ?**

*Note: Enter an integer (e.g. 355)*

## 2.2 Finding the best model

Determine the optimal hyper-parameters for your model from the options listed below:

* `criterion` : [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
* `optimizer` : [Stochastic Gradient Descent (SGD)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)
* `Epochs`: Test with **150**, **250**, **500**, and **1000** epochs.
* `Learning Rate`: Experiment with learning rates of **0.00005**, **0.001**, **1**, and **10**.

**Evaluation**: Assess your model's performance by measuring its accuracy on the test set.

*Note: Run the code `torch.manual_seed(42)` to ensure consistency across all experiments.*

In [ ]:
torch.manual_seed(42)   # Set the seed for reproducibility

### 2.2.1 Automatically Tuning Hyperparameters

In this section, you will automate the process of testing different hyperparameter combinations using a loop, as demonstrated in the lab.

Begin by defining the ranges for the hyperparameters you want to explore:
- **Epochs**: Test with **150**, **250**, **500**, and **1000** epochs.
- **Learning Rate**: Experiment with learning rates of **0.00005**, **0.001**, **1**, and **10**.


In [ ]:
# Your code here
epochs_list = [150, 250, 500, 1000]
learning_rates = [0.00005, 0.001, 1, 10]

Next, create a loop that iterates over the list of learning rates, with an inner loop iterating over the list of number of epochs. Inside the inner loop, initialize (define) the model, and also define the optimizer and the loss function (criterion). Then train the model as demonstrated in the lab, and after the training evaluate its performance to obtain the test accuracy for each model. Ensure that you also display the test loss, as you will need it to answer the upcoming questions.

Here’s a reminder of the criterion and optimizer you should use:
- **Criterion**: [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- **Optimizer**: [Stochastic Gradient Descent (SGD)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)

**Hint**: Make sure to define the criterion and optimizer at every iteration (inside the inner loop), otherwise the model might retain previous training states and produce biased results.

(Optional) You can enhance the code by storing the best model along with its best_accuracy and best_params. This way, you'll have an automatic evaluation at the end to identify the best-performing model.



In [ ]:
# Your code here
import copy  # Added to fix NameError
learning_rates = [0.00005, 0.001, 1, 10]
epochs_list = [150, 250, 500, 1000]

# Step 9: Initialize variables to store the best model
best_accuracy = 0.0
best_model_state = None
best_params = None
results = []

# Nested loop over hyperparameters
hidden_size = 150  # Specified previously
print("\nTesting hyperparameter combinations:")
print("------------------------------------")
for lr in learning_rates:
    for epochs in epochs_list:
        # Initialize a fresh model
        model = CreditMixClassifier(input_size=D_in, hidden_size=hidden_size, output_size=D_out)

        # Define criterion and optimizer
        criterion = nn.CrossEntropyLoss()  # Default reduction='mean'
        optimizer = optim.SGD(model.parameters(), lr=lr)

        # Training loop (without DataLoader)
        model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            outputs = model(X_train_tensor)
            loss = criterion(outputs, y_train_tensor)
            loss.backward()
            optimizer.step()

        # Evaluate test loss and accuracy
        model.eval()
        with torch.no_grad():
            outputs = model(X_test_tensor)
            test_loss = criterion(outputs, y_test_tensor).item()
            _, predicted = torch.max(outputs, 1)
            test_accuracy = (predicted == y_test_tensor).float().mean().item() * 100

        # Store results
        results.append({
            'learning_rate': lr,
            'epochs': epochs,
            'test_accuracy': test_accuracy,
            'test_loss': test_loss
        })
        print(f"Learning Rate: {lr}, Epochs: {epochs}, Test Accuracy: {test_accuracy:.2f}%, Test Loss: {test_loss:.4f}")

        # Update best model
        if test_accuracy > best_accuracy:
            best_accuracy = test_accuracy
            best_model_state = copy.deepcopy(model.state_dict())
            best_params = {'learning_rate': lr, 'epochs': epochs}

# Print results summary
print("\nResults Summary:")
for result in results:
    print(f"LR: {result['learning_rate']}, Epochs: {result['epochs']}, Test Accuracy: {result['test_accuracy']:.2f}%, Test Loss: {test_loss:.4f}")

# Print best model details
print("\nBest Model:")
print(f"Best Test Accuracy: {best_accuracy:.2f}%")
print(f"Best Parameters: Learning Rate = {best_params['learning_rate']}, Epochs = {best_params['epochs']}")

# Calculate and print number of parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\nModel architecture:\n", model)
print("Input size (D_in):", D_in)
print("Hidden size:", hidden_size)
print("Output size (D_out):", D_out)
print("Total number of trainable parameters:", total_params)


Testing hyperparameter combinations:
------------------------------------
Learning Rate: 5e-05, Epochs: 150, Test Accuracy: 30.85%, Test Loss: 1.1006
Learning Rate: 5e-05, Epochs: 250, Test Accuracy: 24.93%, Test Loss: 1.0981
Learning Rate: 5e-05, Epochs: 500, Test Accuracy: 43.54%, Test Loss: 1.0871
Learning Rate: 5e-05, Epochs: 1000, Test Accuracy: 24.28%, Test Loss: 1.1185
Learning Rate: 0.001, Epochs: 150, Test Accuracy: 45.20%, Test Loss: 1.0864
Learning Rate: 0.001, Epochs: 250, Test Accuracy: 45.92%, Test Loss: 1.0774
Learning Rate: 0.001, Epochs: 500, Test Accuracy: 46.47%, Test Loss: 1.0689
Learning Rate: 0.001, Epochs: 1000, Test Accuracy: 45.92%, Test Loss: 1.0419
Learning Rate: 1, Epochs: 150, Test Accuracy: 81.81%, Test Loss: 0.3984
Learning Rate: 1, Epochs: 250, Test Accuracy: 83.52%, Test Loss: 0.3630
Learning Rate: 1, Epochs: 500, Test Accuracy: 83.78%, Test Loss: 0.3485
Learning Rate: 1, Epochs: 1000, Test Accuracy: 83.97%, Test Loss: 0.3369
Learning Rate: 10, Epochs:

### 2.2.3 Questions

**Q2. What is the test accuracy when we train the model with a learning rate of 0.001 and for 150 epochs? Round your answer to 2 decimal points, e.g., 0.25.**


**Q3. When using 1000 epochs, which learning rate results in the highest test accuracy?**

*Note: Select among the following answers*


**Q4. Is BCELoss a suitable alternative to CrossEntropyLoss for our dataset?**

*Hint: Consider the unique values in the Credit_Mix output variable when answering.*

### 3. Predict on the Suspects Dataset

Now it's time to use your trained model to make predictions on the suspects dataset!

Please retrain on the full dataset.

Use the following parameters for the model:
- **Hidden layer**: 1 hidden layer with 150 neurons
- **Output layer**: 3 neurons for classification
- **Optimizer**: [Stochastic Gradient Descent (SGD)](https://en.wikipedia.org/wiki/Stochastic_gradient_descent)
- **Criterion**: [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- **Iterations**: 1000
- **Learning rate**: 1.0

Ensure consistency by setting the manual seed with `torch.manual_seed(42)` before training.

In [ ]:
# Your code here
suspects_encoded = suspects.reindex(columns=X.columns, fill_value=0)

# Normalize using the same scaler
suspects_scaled = scaler.transform(suspects_encoded)

# Convert to PyTorch tensor
suspects_tensor = torch.tensor(suspects_scaled, dtype=torch.float)
model.eval()
with torch.no_grad():
    outputs = model(suspects_tensor)
    _, predicted = torch.max(outputs, 1)
    predicted_classes = predicted.numpy()

# Map numerical predictions back to class names
reverse_mapping = {0: 'Bad', 1: 'Standard', 2: 'Good'}
predicted_labels = [reverse_mapping[pred] for pred in predicted_classes]

# Create a DataFrame with predictions
suspects_predictions = pd.DataFrame({
    'Predicted_Credit_Mix': predicted_labels
}, index=suspects.index)

# Save or display predictions
# suspects_predictions.to_csv('suspects_predictions.csv')  # Uncomment to save
print("\nPredictions on suspects dataset (first 10 rows):")
print(suspects_predictions.head(10))

# Calculate and print number of parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("\nModel architecture:\n", model)
print("Input size (D_in):", D_in)
print("Hidden size:", hidden_size)
print("Output size (D_out):", D_out)
print("Total number of trainable parameters:", total_params)



Now, train your model on the training dataset just as you did in section 2.2.1, but you don't have to loop over all the hyper parameters. Ensure that you use the correct number of epochs specified earlier.

In [ ]:
# Your code here
epochs = 1000
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# Evaluate test loss and accuracy
model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    test_loss = criterion(outputs, y_test_tensor).item()
    _, predicted = torch.max(outputs, 1)
    test_accuracy = (predicted == y_test_tensor).float().mean().item() * 100

print(f"\nTest Accuracy: {test_accuracy:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

Before making predictions, confirm that the feature column names in the `suspects` dataset match those expected by the model, particularly ensuring the X features correspond accurately.

In [ ]:
# Your code here
suspects.rename(columns={"Payment_Behaviour": "le_Payment_Behaviour", "Payment_of_Min_Amount": "le_Payment_of_Min_Amount"}, inplace=True)
# Preprocess suspects dataset
# Define expected features (excluding target and NaN columns)
expected_features = [col for col in df.columns if col != 'Credit_Mix' and col not in ['Changed_Credit_Limit', 'Num_Credit_Inquiries']]

# Apply label encoding
try:
    suspects['le_Payment_of_Min_Amount'] = le_payment_min.transform(suspects['le_Payment_of_Min_Amount'])
    suspects['le_Payment_Behaviour'] = le_payment_behaviour.transform(suspects['le_Payment_Behaviour'])
except ValueError as e:
    print("Error in label encoding suspects:", e)
    print("Training Payment_of_Min_Amount values:", le_payment_min.classes_)
    print("Suspects le_Payment_of_Min_Amount values:", suspects['le_Payment_of_Min_Amount'].unique())
    print("Training Payment_Behaviour values:", le_payment_behaviour.classes_)
    print("Suspects le_Payment_Behaviour values:", suspects['le_Payment_Behaviour'].unique())
    raise

suspects = suspects.drop(['le_Payment_of_Min_Amount', 'le_Payment_Behaviour'], axis=1, errors='ignore')

# Apply one-hot encoding
ohe_encoded_suspects = ohe.transform(suspects[['Occupation']])
suspects_ohe = pd.DataFrame(ohe_encoded_suspects, columns=ohe_columns, index=suspects.index)
suspects = pd.concat([suspects.drop(['Occupation'], axis=1), suspects_ohe], axis=1)

# Confirm feature alignment
missing = [f for f in expected_features if f not in suspects.columns]
extra = [f for f in suspects.columns if f not in expected_features and f != 'Credit_Mix']
if missing or extra:
    print(f"Missing features in suspects dataset: {missing}")
    print(f"Extra features in suspects dataset: {extra}")
else:
    print("Feature names match between training and suspects datasets.")

# Print columns for verification
print("Training dataset columns:", df.columns.tolist())
print("Suspects dataset columns:", suspects.columns.tolist())

Then scale your dataset and convert it into a torch tensor of dtype float, similar to the preprocessing done for the training set in section 1.2.

In [ ]:
#Your code here
numerical_features = [
    'Age', 'Annual_Income', 'Monthly_Inh_Salary', 'Num_Bank_Accounts', 'Num_Credit_Cards',
    'Interest_Rate', 'Num_of_Loans', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
    'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Total_EMI_per_month',
    'Amount_invested_monthly', 'Monthly_Balance'
]
# Scale numerical features in suspects dataset
suspects_encoded[numerical_features] = scaler.transform(suspects_encoded[numerical_features])

# Convert to PyTorch tensor
suspects_tensor = torch.tensor(suspects_encoded[expected_features].values, dtype=torch.float32)

# Verify tensor
print("Suspects tensor shape:", suspects_tensor.shape)
print("Suspects tensor dtype:", suspects_tensor.dtype)

Make predictions using the trained model and assign the predicted credit score to each user. Ensure to do the following encoding is used for the predicted categories:
* `0` corresponds to bad credit score,
* `1` corresponds to standard credit score,
* `2` corresponds to good credit score.

Use the predictions to add a new column `credit_score` in the `suspects` dataframe that maps the predicted numerical values to the respective credit score categories.

In [ ]:
# Your code here



As mentioned earlier, we believe the suspect had a "good" credit score. Review the predictions made by the model, then display how many suspects were categorized under each credit score, and extract the `userID`s of those with a "standard" credit score.

In [ ]:
# Your code here

**Q5.Which of the following suspects have a "good" credit mix according to your model's predictions?**


## Your investigation is progressing effectively, and the list of suspects is narrowing down.

**Don't forget to answer the quiz and submit your code on Moodle before the end of the deadline.**